# Indic translation and speech quality harness

This notebook walks through the harness that evaluates English → Hindi / Tamil / Bengali translations and their speech. It uses **Indic-Translate** (generation and back-translation), **Indic-Speak** (TTS) and **Indic-Transcribe** (STT), plus an optional local Ollama judge.

**Requires** a Bodhan key in `.env` (see `.env.example`) for the API steps. Steps 1 and 2 run offline. For the full explanation see `docs/TECHNICAL_GUIDE.md`.

In [ ]:
%pip install -q -r requirements.txt
import json
import sys

sys.path.insert(0, 'src')

## Step 1: Deterministic checks (offline)

Every check returns `pass`, `fail` (provable error) or `review` (needs a human).

In [ ]:
from bodhan_eval.checks import run_deterministic_checks

record = {
    'id': 'demo', 'language': 'hi',
    'source_text': 'The hospital is 2 kilometers away and opens at 6 PM.',
    'translation': 'अस्पताल 2 मीटर दूर है और सुबह 6 बजे खुलता है।',
}
for check in run_deterministic_checks(record):
    print(check['category'], check['status'], check['evidence'].get('contradictions', ''))

## Step 2: Calibrate the detectors on injected errors (offline)

In [ ]:
from bodhan_eval.calibration import evaluate_injected_cases
from bodhan_eval.pipeline import load_calibration_cases

result = evaluate_injected_cases(load_calibration_cases('sample_data/calibration.jsonl'))
for mode, metrics in result['modes'].items():
    if metrics:
        print(mode, metrics['overall'])

## Step 3: The full pipeline (calls the Bodhan API)

`--back-translate` and `--speech` use your key. Everything is cached, so a second run is free. Add `--llm-judge` if Ollama is running with `llama3.2:3b`.

In [ ]:
!PYTHONPATH=src python -m bodhan_eval --dataset sample_data/demo_100.jsonl --langs hi,ta,bn \
    --out outputs/notebook --calibration sample_data/calibration.jsonl \
    --back-translate --speech --speech-limit 3 --max-cost-inr 20

In [ ]:
summary = json.load(open('outputs/notebook/summary.json'))
print({k: summary[k] for k in ['total_items', 'failed_checks', 'review_checks', 'errors_by_category', 'reviews_by_category']})
print(summary['run']['cost'])

## Wrap-up

You now have `summary.md`, `items.jsonl` and `calibration.json` in `outputs/notebook`. Next steps: add a `glossary` to your records, supply independent `reference_translation`s, or compare runs with `--baseline outputs/notebook/summary.json`.